## Setup
Run notebooks 01–07 first. Install dependencies once from the project root with `python -m pip install -r requirements.txt`.

In [1]:
from pathlib import Path
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate/"data").is_dir() and (candidate/"notebooks").is_dir(): PROJECT_ROOT=candidate; break
else: raise RuntimeError("Start Jupyter inside the project so its root can be found.")
DATA_DIR=PROJECT_ROOT/"data"; RAW_DATA_DIR=DATA_DIR/"01_raw"; INTERMEDIATE_DIR=DATA_DIR/"02_intermediate"
MODEL_DIR=PROJECT_ROOT/"models"; REPORT_DIR=PROJECT_ROOT/"reports"
INTERMEDIATE_DIR.mkdir(parents=True,exist_ok=True); MODEL_DIR.mkdir(parents=True,exist_ok=True); REPORT_DIR.mkdir(parents=True,exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model


# 09 · Model explainability

This notebook explains global patterns and selected individual predictions with SHAP. Attributions describe how inputs move the model output relative to a chosen background. They do not establish causation, prove fraud, or explain information the model never received.

## 1. Load the model and validation rows

In [3]:
!pip install shap==0.41.0 --quiet
import joblib
import numpy as np
import pandas as pd
import shap

train_fe=pd.read_pickle(INTERMEDIATE_DIR/"train_features.pkl"); artifact=joblib.load(MODEL_DIR/"best_experiment_model.joblib")
features=list(artifact["feature_cols"]); ordered=train_fe.sort_values("TransactionDT",kind="stable").reset_index(drop=True)
val=ordered.iloc[int(len(ordered)*0.8):].copy(); X=val[features].replace([np.inf,-np.inf],np.nan)
if artifact.get("imputer") is not None: X=pd.DataFrame(artifact["imputer"].transform(X),columns=features,index=val.index)
if artifact.get("scaler") is not None: X=pd.DataFrame(artifact["scaler"].transform(X),columns=features,index=val.index)
model=artifact["model"]; print(type(model).__name__,X.shape)

  error: subprocess-exited-with-error
  
  × Building wheel for shap (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [117 lines of output]
      The nvcc binary could not be located in your $PATH. Either  add it to your path, or set $CUDAHOME to enable CUDA
      Error building cuda module: TypeError('cannot unpack non-iterable NoneType object')
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-312\shap
      copying shap\datasets.py -> build\lib.win-amd64-cpython-312\shap
      copying shap\links.py -> build\lib.win-amd64-cpython-312\shap
      copying shap\_explanation.py -> build\lib.win-amd64-cpython-312\shap
      copying shap\_serializable.py -> build\lib.win-amd64-cpython-312\shap
      copying shap\__init__.py -> build\lib.win-amd64-cpython-312\shap
      creating build\lib.win-amd64-cpython-312\shap\explainers
      copying shap\explainers\mimic.py -> build\lib.win-amd64-cpython-312\shap\explainers

ModuleNotFoundError: No module named 'shap'

## 2. Build explanations on a fixed, bounded sample

The fixed seed makes the sample repeatable. A modest background and sample keep runtime manageable. The background population defines what “relative to typical” means in these plots.

In [ ]:
SAMPLE_SIZE=min(500,len(X)); BACKGROUND_SIZE=min(100,len(X)); rng=np.random.default_rng(42)
X_sample=X.iloc[rng.choice(len(X),size=SAMPLE_SIZE,replace=False)]
background=X.iloc[rng.choice(len(X),size=BACKGROUND_SIZE,replace=False)]
explainer=shap.Explainer(model,background); explanation=explainer(X_sample)
print("Explanation shape:",explanation.values.shape)

## 3. Global feature influence

In [ ]:
values=explanation
# Classifier explainers may return a final output axis; select fraud class 1.
if values.values.ndim==3:
    base=values.base_values[:,1] if np.ndim(values.base_values)==2 else values.base_values
    values=shap.Explanation(values=values.values[:,:,1],base_values=base,data=values.data,feature_names=values.feature_names)
shap.plots.beeswarm(values,max_display=20)

In [ ]:
shap.plots.bar(values,max_display=20)

## 4. Explain one high-scoring example

In [ ]:
sample_prob=model.predict_proba(X_sample)[:,1]; local=int(np.argmax(sample_prob))
print(f"Fraud probability output: {sample_prob[local]:.4f}")
shap.plots.waterfall(values[local],max_display=15)

## 5. Review limits

Correlated inputs can share attributions. A strong feature may be a proxy or data-quality artifact, so verify its source and availability at decision time. Explanations support analyst review; they are not evidence by themselves. Record model version, explainer version, and reference population when operationalizing explanations.